In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

# Multiple models with caching for feature importance analysis

In [ ]:
import pandas as pd
import seaborn as sns
import utils.base_utils as bu
import utils.window_utils as wu
import numpy as np
from utils.macro_grouping import add_group_level, build_full_group_mapping, groups_as_array

# Bianchi period:
start_date = '1971-08-31'
end_date = '2018-12-31'

# end_date = '2025-06-30' # kr and gsw end date
maturities = [str(i) for i in range(12, 121) if i % 12 == 0] # select only yearly maturities

yields = bu.get_yields(type='lw', start=start_date, end=end_date, maturities=maturities) # type can be kr, lw, gsw
forward = bu.get_forward_rates(yields)
xr = bu.get_excess_returns(yields, horizon=12).dropna() # horizon=12 means holding for 12 months
fred_md_raw = bu.get_fred_data('data/2026-01-MD.csv', start=start_date, end=end_date) # this is aligned to the last day of the previous month, so we get the same number of observations as the yields data

monthly_yields = bu.get_yields(type='lw', start=start_date, end=end_date, maturities=[str(i) for i in range(1, 121)]) # needed for monthly holding period excess returns. Not available for gsw
monthly_xr = bu.get_excess_returns(monthly_yields, horizon=1).dropna() # calculate monthly excess returns for robustness

# At time t (end of month), we only know data for month t-1
fred_md = fred_md_raw.shift(1)  

# Drop dates outside the xr range
yields = yields.loc[yields.index <= xr.index[-1]]
forward = forward.loc[forward.index <= xr.index[-1]]
xr = xr.loc[xr.index <= xr.index[-1]]
fred_md = fred_md.loc[fred_md.index <= xr.index[-1]]
monthly_xr = monthly_xr.loc[monthly_xr.index <= xr.index[-1]]

# Backfill fred_md to avoid nans
fred_md = fred_md.bfill()

# Construct X with 3-level MultiIndex: (source, group, series)
s2g = build_full_group_mapping(fred_md, forward, yields)

X = pd.concat([fred_md, forward, yields],
               axis=1,
               keys=['fred', 'forward', 'yields'])

X = add_group_level(X, s2g, level_name='group')
X = X.sort_index(axis=1, level='group')
groups = groups_as_array(X, level='group')

y_all = monthly_xr[['24','36','48','60','72','84','96','108','120']].values
dates = monthly_xr.index

y = monthly_xr['120'].values # 1-month excess returns
# y = xr['120'].values # 10-year overlapping excess returns
OOS_start = pd.Timestamp('1990-01-31')
# OOS_start = pd.Timestamp('1972-01-31')

In [ ]:
from models.base import *
from models.classical import *
from models.other import *
from models.gbt import *
from models.linear import *
from models.tree import *
from models.ann import *
from utils.persistence_utils import run_expanding_multi_seed

def model_factory(seed):
    return GroupEnsembleANN(archi_group=(1,), archi_fwd=(3,),
                            do_grid_search=True, tune_every=60,
                            epochs=100, patience=10, seed=seed)

seeds = list(range(0, 2))  # 2 seeds

results = run_expanding_multi_seed(
    model_factory=model_factory,
    seeds=seeds,
    X=X,
    y=y_all,                  # can be 1D or multi-output
    dates=dates,
    oos_start=OOS_start,
    run_name="GEANN_2seeds_monthly",
    gap=0,
    refit_freq=1,
    benchmark="hist_mean"
)

print("Done. Saved all refit models for all seeds.")

seeds:   0%|          | 0/2 [00:00<?, ?it/s]

seed 0:   0%|          | 0/336 [00:00<?, ?it/s]

seeds:   0%|          | 0/2 [00:35<?, ?it/s]


KeyboardInterrupt: 